In [2]:
import re
import time

import pandas as pd

from urllib.parse import urlencode

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime

import requests
from bs4 import BeautifulSoup
import os
import time
import random
import logging
import re
import base64
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from fake_useragent import UserAgent
from concurrent.futures import ThreadPoolExecutor, as_completed

### funciones

In [3]:
def construir_url_cp(
    cp,
    tipo_propiedad,
    precio=None,
    porcentaje=0.20,
    accion="for-sale"
):
    """
    Construye una URL de búsqueda por CP y opcionalmente por precio.

    Ejemplos:
    construir_url_cp("01430","casa")

    construir_url_cp("01430","casa",6000000)
    -> min=4200000 max=7800000
    """

    cp = str(cp).zfill(5)

    base = f"https://www.lamudi.com.mx/{tipo_propiedad}/{accion}/"

    params = {"search": cp}

    if precio is not None:
        minimo = int(precio * (1 - porcentaje))
        maximo = int(precio * (1 + porcentaje))

        params["min-price"] = minimo
        params["max-price"] = maximo
        params["priceCurrency"] = "MXN"

    return base + "?" + urlencode(params)

def obtener_nombre_archivo(cp, tipo_propiedad, accion="for-sale"):
    """
    Ejemplo:
    2026-07-14_casa_01430_for_sale.csv
    """

    fecha = datetime.now().strftime("%Y-%m-%d")

    return (
        f"{fecha}_"
        f"{tipo_propiedad}_"
        f"{str(cp).zfill(5)}_"
        f"{accion.replace('-', '_')}.csv"
    )

In [10]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def obtener_listados_v2(cp,tipo_propiedad,precio, max_listados=6):

    start_url = construir_url_cp(cp,tipo_propiedad,precio)
    archivo = obtener_nombre_archivo(cp, tipo_propiedad)


    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/138.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "es-MX,es;q=0.9,en;q=0.8",
    }

    enlaces = []
    visitados = set()

    url = start_url
    pagina = 1

    while url:

        print(f"\nPágina {pagina}")
        print(url)

        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        enlaces_pagina = []

        # Obtener anuncios de la página
        for a in soup.select(".snippet__content a[href]"):
            href = a.get("href")

            if href:
                href = urljoin(url, href)

                if href not in visitados:
                    visitados.add(href)
                    enlaces.append(href)
                    enlaces_pagina.append(href)

        print(f"Links nuevos encontrados: {len(enlaces_pagina)}")
        print(f"Total acumulado: {len(enlaces)}")

        # Ya alcanzamos el máximo
        if len(enlaces) >= max_listados:
            print(f"\nSe alcanzó el máximo solicitado ({max_listados}).")
            return enlaces[:max_listados]

        # Buscar página siguiente
        next_button = soup.select_one("#pagination-next")

        if next_button is None:
            print("\nNo hay más páginas.")
            break

        href = next_button.get("href")

        if not href:
            print("\nEl botón siguiente no tiene href.")
            break

        nueva_url = urljoin(url, href)

        # Evitar ciclos
        if nueva_url == url:
            print("\nLa URL siguiente es la misma. Terminando.")
            break

        url = nueva_url
        pagina += 1

    print(f"\nSolo se encontraron {len(enlaces)} anuncios.")
    return enlaces

In [5]:


# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_stealth_driver():
    """
    Configura el driver con técnicas anti-detección OPTIMIZADO PARA VELOCIDAD
    """
    chrome_options = Options()
    
    # Detectar si estamos en Cloud Run
    is_cloud_run = os.environ.get('K_SERVICE') is not None
    
    # === 1. ARGUMENTOS PARA VELOCIDAD ===
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    
    # OPTIMIZACIONES DE VELOCIDAD
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-plugins")
    chrome_options.add_argument("--disable-images")  # ¡No cargar imágenes!
    chrome_options.add_argument("--disable-javascript")  # ¡NO! Dejar habilitado
    
    # Deshabilitar características que ralentizan
    chrome_options.add_argument("--disable-background-timer-throttling")
    chrome_options.add_argument("--disable-backgrounding-occluded-windows")
    chrome_options.add_argument("--disable-renderer-backgrounding")
    chrome_options.add_argument("--disable-application-cache")
    chrome_options.add_argument("--disable-cache")  # No usar caché
    chrome_options.add_argument("--disk-cache-size=0")  # Sin caché en disco
    chrome_options.add_argument("--media-cache-size=0")  # Sin caché de medios
    
    # Anti-detección
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # User-Agent
    try:
        ua = UserAgent()
        chrome_options.add_argument(f'user-agent={ua.random}')
    except:
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # Configuración por entorno
    if is_cloud_run:
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--remote-debugging-port=9222")
        service = Service('/usr/bin/chromedriver')
    else:
        # En local, puedes ver el navegador para depurar
        # chrome_options.add_argument("--headless=new")  # Comenta para ver
        service = Service(ChromeDriverManager().install())
    
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    # Scripts de evasión
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': '''
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
            Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3, 4, 5] });
            Object.defineProperty(navigator, 'languages', { get: () => ['es-ES', 'es'] });
            Object.defineProperty(navigator, 'headless', { get: () => false });
            window.chrome = { runtime: {} };
            delete window.document.$cdc_asdjflasutopfhvcZLmcfl_;
        '''
    })
    
    # Timeouts reducidos para mayor velocidad
    driver.set_page_load_timeout(15)  # 15 segundos máximo para cargar
    driver.set_script_timeout(10)
    
    logger.info("✅ Driver optimizado iniciado")
    return driver

def scrape_property_fast(url):
    """
    Versión RÁPIDA de scraping para una sola propiedad
    """
    driver = None
    data = {
        'titulo': None,
        'direccion': None,
        'tipo': None,
        'precio': None,
        'superficie': None,
        'habitaciones': None,
        'banios': None,
        'caracteristica_propiedad': None,
        'amenidades': None,
        'caracteristicas': None,
        'planta': None,
        'descripcion': None,
        'fecha_publicacion': None,
        'url': url,
        'lat': None,
        'lon': None,
        'script_content': None,
        'fecha_consulta': time.strftime("%Y-%m-%d"),
        'error': False
    }
    
    try:
        start_time = time.time()
        logger.info(f"⬇️ Scrapeando: {url}")
        
        driver = get_stealth_driver()
        
        # === 1. NAVEGACIÓN RÁPIDA ===
        driver.get(url)
        
        # === 2. ESPERA MÍNIMA ===
        # En lugar de sleep fijo, esperar solo lo necesario
        try:
            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )
        except:
            # Si no encuentra h1 en 5 segundos, continuar igual
            pass
        
        # === 3. SCROLL MÍNIMO (solo uno, rápido) ===
        driver.execute_script("window.scrollTo(0, 300);")
        time.sleep(0.5)  # Sleep reducido
        
        # === 4. VERIFICAR BLOQUEO RÁPIDO ===
        body_text = driver.find_element(By.TAG_NAME, "body").text
        if "403" in body_text or "404" in body_text or "ERROR" in body_text:
            logger.error(f"❌ Página bloqueada: {url}")
            data['error'] = True
            return data
        
        # === 5. SELECTORES MÚLTIPLES (optimizados) ===
        selectores = {
            'titulo': [
                (By.TAG_NAME, "h1"),
                (By.CSS_SELECTOR, ".property-title")
            ],
            'direccion': [
                (By.CSS_SELECTOR, "div.location-map__location-address-map"),
                (By.CSS_SELECTOR, ".address")
            ],
            'tipo': [
                (By.CSS_SELECTOR, ".property-type span.place-features__values"),
                (By.CSS_SELECTOR, ".property-type")
            ],
            'precio': [
                (By.CSS_SELECTOR, "div.prices-and-fees__price"),
                (By.CSS_SELECTOR, ".price")
            ],
            'superficie': [
                (By.CSS_SELECTOR, "div.details-item-value[data-test='area-value']"),
                (By.CSS_SELECTOR, ".area")
            ],
            'habitaciones': [
                (By.CSS_SELECTOR, "div.details-item-value[data-test='bedrooms-value']"),
                (By.CSS_SELECTOR, ".bedrooms")
            ],
            'banios': [
                (By.CSS_SELECTOR, "div.details-item-value[data-test='full-bathrooms-value']"),
                (By.CSS_SELECTOR, ".bathrooms")
            ],
            'descripcion': [
                (By.CSS_SELECTOR, "div#description-text"),
                (By.CSS_SELECTOR, ".description")
            ],
            'fecha_publicacion': [
                (By.CSS_SELECTOR, "div.date"),
                (By.CSS_SELECTOR, ".publish-date")
            ]
        }
        
        # Extraer datos
        for key, selectors_list in selectores.items():
            for by, selector in selectors_list:
                try:
                    elemento = driver.find_element(by, selector)
                    if elemento:
                        data[key] = elemento.text.strip()
                        break
                except:
                    continue
        
        # === 6. COORDENADAS (solo si es necesario) ===
        try:
            scripts = driver.find_elements(By.TAG_NAME, "script")
            for script in scripts[:5]:  # Solo revisar primeros 5 scripts
                txt = script.get_attribute("innerHTML")
                if txt and ("var pageData" in txt or "latitude" in txt):
                    data['script_content'] = txt
                    lat_match = re.search(r'latitude:\s*"?([-\d.]+)"?', txt)
                    lon_match = re.search(r'longitude:\s*"?([-\d.]+)"?', txt)
                    if lat_match:
                        data['lat'] = float(lat_match.group(1))
                    if lon_match:
                        data['lon'] = float(lon_match.group(1))
                    break
        except:
            pass
        
        elapsed = time.time() - start_time
        logger.info(f"✅ Completado en {elapsed:.1f}s: {data['titulo'][:30] if data['titulo'] else 'Sin título'}")
        
    except Exception as e:
        logger.error(f"❌ Error en {url}: {str(e)}")
        data['error'] = True
    
    finally:
        if driver:
            driver.quit()
    
    return data

def scrape_listados_parallel(property_links, output_filename=None, max_workers=3):
    """
    Versión PARALELIZADA para múltiples URLs
    """
    if not property_links:
        return pd.DataFrame()
    
    logger.info(f"🚀 Iniciando scraping paralelo de {len(property_links)} URLs con {max_workers} workers")
    
    all_data = []
    
    # Usar ThreadPoolExecutor para paralelizar
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Enviar todas las tareas
        future_to_url = {executor.submit(scrape_property_fast, url): url for url in property_links}
        
        # Recoger resultados a medida que se completan
        for future in as_completed(future_to_url):
            url = future_to_url[future]
            try:
                data = future.result(timeout=30)
                all_data.append(data)
                logger.info(f"📊 Progreso: {len(all_data)}/{len(property_links)} completadas")
            except Exception as e:
                logger.error(f"❌ Error en {url}: {e}")
                # Agregar registro de error
                all_data.append({
                    'url': url,
                    'error': True,
                    'fecha_consulta': time.strftime("%Y-%m-%d")
                })
    
    # Crear DataFrame
    df = pd.DataFrame(all_data)
    
    # Guardar CSV
    if output_filename:
        df.to_csv(output_filename, index=False, encoding='utf-8')
        logger.info(f"💾 Datos guardados en {output_filename}")
    
    # Estadísticas
    total = len(df)
    errores = df[df.get('error', False) == True].shape[0]
    exitos = total - errores
    logger.info(f"📊 Resumen: {exitos} éxitos, {errores} errores de {total} total")
    
    return df

def scrape_listados_sequential(property_links, output_filename=None):
    """
    Versión SECUENCIAL (más lenta pero más estable)
    """
    logger.info(f"🔄 Iniciando scraping secuencial de {len(property_links)} URLs")
    
    all_data = []
    
    for i, url in enumerate(property_links, 1):
        logger.info(f"\n{'='*50}\n[{i}/{len(property_links)}] Procesando: {url}\n{'='*50}")
        
        # Delay entre peticiones (reducido)
        if i > 1:
            time.sleep(random.uniform(1, 2))
        
        data = scrape_property_fast(url)
        all_data.append(data)
    
    df = pd.DataFrame(all_data)
    
    if output_filename:
        df.to_csv(output_filename, index=False, encoding='utf-8')
        logger.info(f"💾 Datos guardados en {output_filename}")
    
    return df

# === FUNCIÓN PRINCIPAL (elige el modo) ===
def scrape_listados_cloudrun(property_links, output_filename=None, mode='parallel'):
    """
    Scrapea múltiples propiedades
    
    Args:
        property_links: Lista de URLs
        output_filename: Nombre del archivo CSV
        mode: 'parallel' (rápido) o 'sequential' (estable)
    """
    if mode == 'parallel':
        # Paralelo: 2-3 workers es óptimo para Cloud Run
        return scrape_listados_parallel(property_links, output_filename, max_workers=3)
    else:
        # Secuencial: más estable pero más lento
        return scrape_listados_sequential(property_links, output_filename)


In [11]:
### PIPELINE LAMUDI
#variables
# TIPOS_PROPIEDAD = ['casa', 'departamento', 'terreno', 'comercial']
cp= "03100"
tipo_propiedad="departamento"
precio=6000000


links2 = obtener_listados_v2(cp,tipo_propiedad,precio, max_listados=6)
print(links2)


Página 1
https://www.lamudi.com.mx/departamento/for-sale/?search=03100&min-price=4800000&max-price=7200000&priceCurrency=MXN
Links nuevos encontrados: 30
Total acumulado: 30

Se alcanzó el máximo solicitado (6).
['https://www.lamudi.com.mx/detalle/41032-73-ef6d92a39427-8219-19d4c1f-9ec1-74ce', 'https://www.lamudi.com.mx/detalle/41032-73-1382cbb62938-13c4-19e419a-9e76-7439', 'https://www.lamudi.com.mx/detalle/41032-73-771e45cd24bc-2bc2-19f043b-a1da-7bf5', 'https://www.lamudi.com.mx/detalle/41032-73-49d8b16ce0d7-4ba-19dc268-a9ed-7645', 'https://www.lamudi.com.mx/detalle/41032-73-20ee19086e0d-15d5-19dc268-bece-7efe', 'https://www.lamudi.com.mx/detalle/41032-73-f4a85d54614f-df73-18fd185-aa30-7c06']


In [9]:
df = scrape_listados_cloudrun(links2,archivo)

INFO:__main__:🚀 Iniciando scraping paralelo de 6 URLs con 3 workers
INFO:__main__:⬇️ Scrapeando: https://www.lamudi.com.mx/detalle/41032-73-ef6d92a39427-8219-19d4c1f-9ec1-74ce
INFO:__main__:⬇️ Scrapeando: https://www.lamudi.com.mx/detalle/41032-73-1382cbb62938-13c4-19e419a-9e76-7439
INFO:__main__:⬇️ Scrapeando: https://www.lamudi.com.mx/detalle/41032-73-771e45cd24bc-2bc2-19f043b-a1da-7bf5
INFO:WDM:====== WebDriver manager ======
INFO:WDM:====== WebDriver manager ======
INFO:WDM:====== WebDriver manager ======
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Driver [C:\Users\THUNDEROBOT\.wdm\drivers\chromedriver\win64\150.0.7871.124\chromedriver-win32/chromedriver.exe] found in 